In [1]:
import numpy as np
import pandas as pd


class UserBasedCF:

    def __init__(
        self,
        user_item_matrix,
        top_k=20,
        min_common=10,
        significance_threshold=50
    ):

        self.user_item = user_item_matrix

        self.top_k = top_k
        self.min_common = min_common
        self.significance_threshold = significance_threshold
        # ---------- NumPy matrix ----------

        self.matrix = self.user_item.to_numpy()

        self.user_to_idx = {
            user: i
            for i, user in enumerate(self.user_item.index)
        }

        self.movie_to_idx = {
            movie: j
            for j, movie in enumerate(self.user_item.columns)
        }

        # ---------- User Means ----------

        self.user_means = self.user_item.mean(axis=1)
        self.user_mean_array = self.user_means.to_numpy()

        # ---------- Neighbor Cache ----------

        self.neighbor_cache = {}

    # ==========================================================
    # Pearson Similarity
    # ==========================================================

    def pearson_similarity(
        self,
        user1,
        user2
    ):

        common = user1.notna() & user2.notna()

        common_count = common.sum()

        if common_count < self.min_common:
            return 0.0, common_count

        u1 = user1[common]
        u2 = user2[common]

        u1 = u1 - u1.mean()
        u2 = u2 - u2.mean()

        norm1 = np.linalg.norm(u1)
        norm2 = np.linalg.norm(u2)

        if norm1 == 0 or norm2 == 0:
            return 0.0, common_count

        similarity = np.dot(u1, u2) / (norm1 * norm2)
        weight=min(common_count, self.significance_threshold) / self.significance_threshold
        similarity *= weight
        return similarity, common_count

    # ==========================================================
    # Find Neighbors
    # ==========================================================

    def get_neighbors(
        self,
        target_user,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        if target_user in self.neighbor_cache:
            return self.neighbor_cache[target_user]

        target = self.user_item.loc[target_user]

        neighbors = []

        for other_user in self.user_item.index:

            if other_user == target_user:
                continue

            similarity, common = self.pearson_similarity(
                target,
                self.user_item.loc[other_user]
            )

            if similarity <= 0:
                continue

            neighbors.append(
                (
                    other_user,
                    similarity,
                    common
                )
            )

        neighbors.sort(
            key=lambda x: x[1],
            reverse=True
        )

        neighbors = neighbors[:top_k]

        self.neighbor_cache[target_user] = neighbors

        return neighbors

    # ==========================================================
    # Candidate Movies
    # ==========================================================

    def get_candidate_movies(
        self,
        target_user,
        top_k=None
    ):

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        watched_movies = set(
            self.user_item.loc[target_user]
            .dropna()
            .index
        )

        candidate_movies = set()

        for neighbor_id, similarity, common in neighbors:

            neighbor_movies = (
                self.user_item
                .loc[neighbor_id]
                .dropna()
                .index
            )

            candidate_movies.update(
                neighbor_movies
            )

        candidate_movies -= watched_movies

        return list(candidate_movies)
    
    # ==========================================================
    # Predict Rating
    # ==========================================================

    def predict_rating(
        self,
        target_user,
        target_movie,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        # Movie not present in training
        if target_movie not in self.movie_to_idx:
            return None

        neighbors = self.get_neighbors(
            target_user,
            top_k
        )

        target_idx = self.user_to_idx[target_user]
        movie_idx = self.movie_to_idx[target_movie]

        target_mean = self.user_mean_array[target_idx]

        numerator = 0.0
        denominator = 0.0

        for neighbor_id, similarity, common in neighbors:

            neighbor_idx = self.user_to_idx[neighbor_id]

            rating = self.matrix[
                neighbor_idx,
                movie_idx
            ]

            if np.isnan(rating):
                continue

            neighbor_mean = self.user_mean_array[
                neighbor_idx
            ]

            numerator += similarity * (
                rating - neighbor_mean
            )

            denominator += similarity

        if denominator == 0:
            return None

        prediction = target_mean + (
            numerator / denominator
        )

        prediction = np.clip(
            prediction,
            1,
            5
        )

        return float(prediction)


    # ==========================================================
    # Recommend Movies
    # ==========================================================

    def recommend(
        self,
        target_user,
        top_n=10,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        recommendations = []

        candidate_movies = self.get_candidate_movies(
            target_user,
            top_k
        )

        for movie in candidate_movies:

            prediction = self.predict_rating(
                target_user,
                movie,
                top_k
            )

            if prediction is None:
                continue

            recommendations.append(
                (
                    movie,
                    prediction
                )
            )

        recommendations.sort(
            key=lambda x: x[1],
            reverse=True
        )

        recommendations = recommendations[:top_n]

        return pd.DataFrame(
            recommendations,
            columns=[
                "movie_id",
                "predicted_rating"
            ]
        )


    # ==========================================================
    # Evaluate
    # ==========================================================

    def evaluate(
        self,
        test_df
    ):

        predictions = []

        squared_errors = []

        absolute_errors = []

        for row in test_df.itertuples(index=False):

            user = row.user_id
            movie = row.movie_id
            actual = row.rating

            predicted = self.predict_rating(
                user,
                movie
            )

            if predicted is None:
                continue

            error = abs(
                actual - predicted
            )

            predictions.append(
                (
                    user,
                    movie,
                    actual,
                    predicted,
                    error
                )
            )

            absolute_errors.append(
                error
            )

            squared_errors.append(
                error ** 2
            )

        prediction_df = pd.DataFrame(
            predictions,
            columns=[
                "user_id",
                "movie_id",
                "actual",
                "predicted",
                "error"
            ]
        )

        mae = np.mean(
            absolute_errors
        )

        rmse = np.sqrt(
            np.mean(
                squared_errors
            )
        )

        return (
            prediction_df,
            rmse,
            mae
        )

In [2]:
train = pd.read_csv(
    "data/u1.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "data/u1.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [3]:
train_matrix = train.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

In [4]:
cf = UserBasedCF(
    train_matrix,
    top_k=20,
    min_common=10
)

In [5]:
cf.get_neighbors(1)

[(880, np.float64(0.5769851034844081), np.int64(67)),
 (892, np.float64(0.5395378418082085), np.int64(49)),
 (592, np.float64(0.5158611292334155), np.int64(69)),
 (650, np.float64(0.49794680641645006), np.int64(64)),
 (886, np.float64(0.4865078634708828), np.int64(56)),
 (868, np.float64(0.4819839316365741), np.int64(48)),
 (452, np.float64(0.46616882364578044), np.int64(46)),
 (303, np.float64(0.45873212759869436), np.int64(52)),
 (429, np.float64(0.4568421029661331), np.int64(71)),
 (815, np.float64(0.45667055802320444), np.int64(44)),
 (660, np.float64(0.44405907257729765), np.int64(52)),
 (773, np.float64(0.4354907706961544), np.int64(40)),
 (682, np.float64(0.43496131594485904), np.int64(77)),
 (790, np.float64(0.4347198150914971), np.int64(49)),
 (13, np.float64(0.43316847635351374), np.int64(50)),
 (291, np.float64(0.42511721983346673), np.int64(32)),
 (479, np.float64(0.42429110678551774), np.int64(52)),
 (653, np.float64(0.41510525977826773), np.int64(57)),
 (293, np.float64(0

In [6]:
cf.predict_rating(
    target_user=1,
    target_movie=300
)

3.457790652281864

In [7]:
import time
start = time.time()
recommendations = cf.recommend(
    target_user=1,
    top_n=10
)
print("Time taken for recommendations: ", time.time() - start)
recommendations

Time taken for recommendations:  0.03406500816345215


,movie_id,predicted_rating
0,12,5.0
1,114,5.0
2,170,5.0
3,206,5.0
4,279,5.0
5,408,5.0
6,492,5.0
7,511,5.0
8,632,5.0
9,646,5.0


In [8]:

start = time.time()
prediction_df, rmse, mae = cf.evaluate(test)
print("Time taken for evaluation: ", time.time() - start)

print("MAE :", mae)
print("RMSE:", rmse)

prediction_df.head()

Time taken for evaluation:  95.0829427242279
MAE : 0.7667065861805898
RMSE: 0.9933578909326893


,user_id,movie_id,actual,predicted,error
0,1,6,5,3.401978,1.598022
1,1,10,3,2.538087,0.461913
2,1,12,5,5.000000,0.000000
3,1,14,5,4.291012,0.708988
4,1,17,3,3.195739,0.195739
